# The graphical representation of the optimization worflow

In [2]:
"""
TKFRONT GUI for Eclipse + Optuna optimization

What it does
- Lets you set search-space bounds for: BHP1-BHP4 (4 producers), inj_rate1-inj_rate8 (8 injectors)
- Lets you choose optimization algorithm:
    * TPE (single-objective: maximize FOPT)
    * NSGA-II (multi-objective: maximize FOPT, minimize FWPT)
- Lets you set number of trials + seed
- Runs optimization and shows best results / Pareto front summary

Requirements
- pip install optuna
- Eclipse runnable via ECLRUN_EXE and files in BASE_DIR
"""

import os
import threading
import subprocess
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import optuna


# ----------------------------
# Defaults (your paths)
# ----------------------------
ECLRUN_EXE = r"C:\ecl\macros\eclrun.exe"
BASE_DIR = r"C:\Users\hp\Desktop\Eclipse"
DATA_FILE_NAME = "Egg_Model_ECL.DATA"
SCHEDULE_FILE_NAME = "SCHEDULE_NEW.INC"
RSM_PATH = os.path.join(BASE_DIR, "EGG_MODEL_ECL_OVERALL.RSM")

N_INJ = 8
N_PROD = 4


# ----------------------------
# Core simulation functions
# ----------------------------
def modify_files(params):
    # --------------------------------------------------
    # Modify SCHEDULE file (Injector RATES) — 8 injectors
    # --------------------------------------------------
    sched_path = os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)

    inj_map = {
        "INJECT1": params["inj_rate1"],
        "INJECT2": params["inj_rate2"],
        "INJECT3": params["inj_rate3"],
        "INJECT4": params["inj_rate4"],
        "INJECT5": params["inj_rate5"],
        "INJECT6": params["inj_rate6"],
        "INJECT7": params["inj_rate7"],
        "INJECT8": params["inj_rate8"],
    }

    with open(sched_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()
        for well, rate in inj_map.items():
            if stripped.startswith(f"'{well}'") and "'RATE'" in stripped:
                parts = stripped.split()
                if "'RATE'" in parts:
                    rate_idx = parts.index("'RATE'") + 1
                    if rate_idx < len(parts):
                        parts[rate_idx] = str(int(rate))
                        line = " ".join(parts) + "\n"
        new_lines.append(line)

    with open(sched_path, "w") as f:
        f.writelines(new_lines)

    # --------------------------------------------------
    # Modify DATA file (Producer BHP) — 4 producers
    # --------------------------------------------------
    data_path = os.path.join(BASE_DIR, DATA_FILE_NAME)

    prod_map = {
        "PROD1": params["BHP1"],
        "PROD2": params["BHP2"],
        "PROD3": params["BHP3"],
        "PROD4": params["BHP4"],
    }

    with open(data_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()
        for well, bhp in prod_map.items():
            if stripped.startswith(f"'{well}'") and "'BHP'" in stripped:
                parts = stripped.split()
                # last token ends with "/" in your file
                parts[-1] = f"{int(bhp)}/"
                line = " ".join(parts) + "\n"
        new_lines.append(line)

    with open(data_path, "w") as f:
        f.writelines(new_lines)


def run_eclipse():
    cmd = [ECLRUN_EXE, "eclipse", DATA_FILE_NAME]
    subprocess.run(cmd, cwd=BASE_DIR, check=True)


def extract_FOPT_FWPT_from_RSM(rsm_path):
    with open(rsm_path, "r") as f:
        lines = f.readlines()

    inside_summary = False
    fopt_idx = fwpt_idx = None

    for line in lines:
        if "SUMMARY OF RUN" in line:
            inside_summary = True
            fopt_idx = fwpt_idx = None
            continue

        if not inside_summary:
            continue

        tokens = line.split()
        if not tokens:
            continue

        # header detection
        if fopt_idx is None and "FOPT" in tokens and "FWPT" in tokens:
            fopt_idx = tokens.index("FOPT")
            fwpt_idx = tokens.index("FWPT")
            continue

        if fopt_idx is None:
            continue

        # numeric rows begin with time
        if not tokens[0][0].isdigit():
            continue

        if float(tokens[0]) == 3600.0:
            return float(tokens[fopt_idx]), float(tokens[fwpt_idx])

    return None, None


# ----------------------------
# Optuna objective factory
# ----------------------------
def make_objective(search_space, mode):
    """
    mode:
      - "tpe_single"  : return fopt (maximize)
      - "nsga2_multi" : return (fopt, fwpt) (maximize, minimize)
    """
    def objective(trial):
        # 4 producer BHP controls
        params = {
            "BHP1": trial.suggest_int("BHP1", search_space["BHP1_min"], search_space["BHP1_max"]),
            "BHP2": trial.suggest_int("BHP2", search_space["BHP2_min"], search_space["BHP2_max"]),
            "BHP3": trial.suggest_int("BHP3", search_space["BHP3_min"], search_space["BHP3_max"]),
            "BHP4": trial.suggest_int("BHP4", search_space["BHP4_min"], search_space["BHP4_max"]),
            # 8 injector rate controls
            "inj_rate1": trial.suggest_int("inj_rate1", search_space["inj1_min"], search_space["inj1_max"]),
            "inj_rate2": trial.suggest_int("inj_rate2", search_space["inj2_min"], search_space["inj2_max"]),
            "inj_rate3": trial.suggest_int("inj_rate3", search_space["inj3_min"], search_space["inj3_max"]),
            "inj_rate4": trial.suggest_int("inj_rate4", search_space["inj4_min"], search_space["inj4_max"]),
            "inj_rate5": trial.suggest_int("inj_rate5", search_space["inj5_min"], search_space["inj5_max"]),
            "inj_rate6": trial.suggest_int("inj_rate6", search_space["inj6_min"], search_space["inj6_max"]),
            "inj_rate7": trial.suggest_int("inj_rate7", search_space["inj7_min"], search_space["inj7_max"]),
            "inj_rate8": trial.suggest_int("inj_rate8", search_space["inj8_min"], search_space["inj8_max"]),
        }

        modify_files(params)

        try:
            run_eclipse()
        except subprocess.CalledProcessError:
            if mode == "nsga2_multi":
                return -1e12, 1e12
            return -1e12

        fopt, fwpt = extract_FOPT_FWPT_from_RSM(RSM_PATH)
        if fopt is None or fwpt is None:
            if mode == "nsga2_multi":
                return -1e12, 1e12
            return -1e12

        if mode == "nsga2_multi":
            return fopt, fwpt
        return fopt

    return objective


# ----------------------------
# Optuna runner
# ----------------------------
def run_optuna(search_space, algo, n_trials, seed, log_fn, stop_flag):
    """
    algo:
      - "TPE (single: max FOPT)"
      - "NSGA-II (multi: max FOPT, min FWPT)"
    """
    if algo.startswith("TPE"):
        sampler = optuna.samplers.TPESampler(
            n_startup_trials=10,
            multivariate=True,
            seed=seed,
        )
        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            study_name="ECLIPSE_TPE_FOPT",
        )
        objective = make_objective(search_space, mode="tpe_single")

    else:
        sampler = optuna.samplers.NSGAIISampler(
            population_size=20,
            mutation_prob=0.2,
            crossover_prob=0.9,
            seed=seed,
        )
        study = optuna.create_study(
            directions=["maximize", "minimize"],
            sampler=sampler,
            study_name="ECLIPSE_NSGA2_FOPT_FWPT",
        )
        objective = make_objective(search_space, mode="nsga2_multi")

    def callback(study, trial):
        if stop_flag["stop"]:
            raise optuna.exceptions.StudyStop()
        if trial.values is None:
            return
        if len(trial.values) == 1:
            log_fn(f"Trial {trial.number:4d}: FOPT={trial.values[0]:.6g}  params={trial.params}\n")
        else:
            log_fn(
                f"Trial {trial.number:4d}: FOPT={trial.values[0]:.6g}  "
                f"FWPT={trial.values[1]:.6g}  params={trial.params}\n"
            )

    log_fn(f"Starting optimization: algo={algo}, n_trials={n_trials}, seed={seed}\n")
    try:
        study.optimize(objective, n_trials=n_trials, callbacks=[callback], show_progress_bar=False)
    except optuna.exceptions.StudyStop:
        log_fn("Stopped by user.\n")
    except Exception as e:
        log_fn(f"ERROR: {e}\n")
        raise

    # summarize
    if study.best_trials:
        log_fn("\n=== Results ===\n")
        if algo.startswith("TPE"):
            bt = study.best_trial
            log_fn(f"Best FOPT : {bt.value:.6g}\n")
            log_fn(f"Best params: {bt.params}\n")
        else:
            pareto = study.best_trials
            log_fn(f"Pareto front size: {len(pareto)}\n")
            for i, t in enumerate(pareto[:10], start=1):
                log_fn(
                    f"Pareto {i:02d}: FOPT={t.values[0]:.6g}  "
                    f"FWPT={t.values[1]:.6g}  params={t.params}\n"
                )

    return study


# ----------------------------
# TKFRONT GUI
# ----------------------------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Eclipse Optimization - TKFRONT")
        self.geometry("820x780")
        self.minsize(820, 600)

        self.stop_flag = {"stop": False}
        self.worker = None

        # ── Paths ──────────────────────────────────────────────────
        paths = ttk.LabelFrame(self, text="Paths")
        paths.pack(fill="x", padx=10, pady=6)

        self.eclrun_var  = tk.StringVar(value=ECLRUN_EXE)
        self.base_var    = tk.StringVar(value=BASE_DIR)
        self.data_var    = tk.StringVar(value=DATA_FILE_NAME)
        self.sched_var   = tk.StringVar(value=SCHEDULE_FILE_NAME)
        self.rsm_var     = tk.StringVar(value=RSM_PATH)

        self._row_entry(paths, 0, "ECLRUN_EXE:",        self.eclrun_var, browse=True, browse_type="file")
        self._row_entry(paths, 1, "BASE_DIR:",          self.base_var,  browse=True, browse_type="dir")
        self._row_entry(paths, 2, "DATA_FILE_NAME:",    self.data_var)
        self._row_entry(paths, 3, "SCHEDULE_FILE_NAME:",self.sched_var)
        self._row_entry(paths, 4, "RSM_PATH:",          self.rsm_var,   browse=True, browse_type="file")

        # ── Search Space ───────────────────────────────────────────
        space = ttk.LabelFrame(self, text="Search Space (integer bounds)")
        space.pack(fill="x", padx=10, pady=6)

        # 4 producer BHP bounds
        self.bhp_min = [tk.IntVar(value=290) for _ in range(N_PROD)]
        self.bhp_max = [tk.IntVar(value=370) for _ in range(N_PROD)]

        # 8 injector rate bounds
        self.inj_min = [tk.IntVar(value=70)  for _ in range(N_INJ)]
        self.inj_max = [tk.IntVar(value=290) for _ in range(N_INJ)]

        # Producer BHP rows
        prod_frame = ttk.LabelFrame(space, text="Producers — BHP (bar)")
        prod_frame.pack(fill="x", padx=6, pady=4)
        for i in range(N_PROD):
            self._bounds_row(prod_frame, i, f"BHP{i+1}", self.bhp_min[i], self.bhp_max[i])

        # Injector rate rows
        inj_frame = ttk.LabelFrame(space, text="Injectors — Rate (sm³/day)")
        inj_frame.pack(fill="x", padx=6, pady=4)
        for i in range(N_INJ):
            self._bounds_row(inj_frame, i, f"INJ_RATE{i+1}", self.inj_min[i], self.inj_max[i])

        # ── Optimization Settings ──────────────────────────────────
        opt = ttk.LabelFrame(self, text="Optimization Settings")
        opt.pack(fill="x", padx=10, pady=6)

        ttk.Label(opt, text="Algorithm:").grid(row=0, column=0, sticky="e", padx=6, pady=6)
        self.algo_var = tk.StringVar(value="NSGA-II (multi: max FOPT, min FWPT)")
        ttk.Combobox(
            opt, textvariable=self.algo_var, state="readonly", width=38,
            values=[
                "TPE (multi: max FOPT, min FWPT)",
                "NSGA-II (multi: max FOPT, min FWPT)",
            ],
        ).grid(row=0, column=1, sticky="w", padx=6, pady=6)

        self.trials_var = tk.IntVar(value=110)
        self.seed_var   = tk.IntVar(value=42)

        ttk.Label(opt, text="n_trials:").grid(row=1, column=0, sticky="e", padx=6, pady=6)
        ttk.Entry(opt, textvariable=self.trials_var, width=12).grid(row=1, column=1, sticky="w", padx=6, pady=6)
        ttk.Label(opt, text="seed:").grid(row=1, column=2, sticky="e", padx=6, pady=6)
        ttk.Entry(opt, textvariable=self.seed_var,   width=12).grid(row=1, column=3, sticky="w", padx=6, pady=6)
        opt.grid_columnconfigure(4, weight=1)

        # ── Buttons ────────────────────────────────────────────────
        btns = ttk.Frame(self)
        btns.pack(fill="x", padx=10, pady=6)

        self.run_btn  = ttk.Button(btns, text="Run Optimization", command=self.on_run)
        self.run_btn.pack(side="left")
        self.stop_btn = ttk.Button(btns, text="Stop", command=self.on_stop, state="disabled")
        self.stop_btn.pack(side="left", padx=10)
        self.clear_btn = ttk.Button(btns, text="Clear Log", command=self.clear_log)
        self.clear_btn.pack(side="left")

        # ── Log ────────────────────────────────────────────────────
        logf = ttk.LabelFrame(self, text="Log")
        logf.pack(fill="both", expand=True, padx=10, pady=6)

        sb = ttk.Scrollbar(logf)
        sb.pack(side="right", fill="y")
        self.log = tk.Text(logf, wrap="word", yscrollcommand=sb.set)
        self.log.pack(fill="both", expand=True)
        sb.config(command=self.log.yview)

        self.log_write("Ready.\n")

    # ── Helpers ────────────────────────────────────────────────────
    def _row_entry(self, parent, row, label, var, browse=False, browse_type="file"):
        ttk.Label(parent, text=label).grid(row=row, column=0, sticky="e", padx=6, pady=3)
        ttk.Entry(parent, textvariable=var, width=80).grid(row=row, column=1, sticky="we", padx=6, pady=3)
        parent.grid_columnconfigure(1, weight=1)
        if browse:
            def do_browse(v=var, bt=browse_type):
                path = filedialog.askopenfilename() if bt == "file" else filedialog.askdirectory()
                if path:
                    v.set(path)
            ttk.Button(parent, text="Browse", command=do_browse).grid(row=row, column=2, padx=6, pady=3)

    def _bounds_row(self, parent, row, name, vmin, vmax):
        ttk.Label(parent, text=f"{name} min:").grid(row=row, column=0, sticky="e", padx=6, pady=3)
        ttk.Entry(parent, textvariable=vmin, width=10).grid(row=row, column=1, sticky="w", padx=6, pady=3)
        ttk.Label(parent, text=f"{name} max:").grid(row=row, column=2, sticky="e", padx=6, pady=3)
        ttk.Entry(parent, textvariable=vmax, width=10).grid(row=row, column=3, sticky="w", padx=6, pady=3)
        parent.grid_columnconfigure(4, weight=1)

    def log_write(self, msg: str):
        self.log.insert("end", msg)
        self.log.see("end")
        self.update_idletasks()

    def clear_log(self):
        self.log.delete("1.0", "end")

    def on_stop(self):
        self.stop_flag["stop"] = True
        self.log_write("\nStop requested...\n")

    # ── Run ────────────────────────────────────────────────────────
    def on_run(self):
        # apply globals from UI
        global ECLRUN_EXE, BASE_DIR, DATA_FILE_NAME, SCHEDULE_FILE_NAME, RSM_PATH
        ECLRUN_EXE       = self.eclrun_var.get().strip()
        BASE_DIR         = self.base_var.get().strip()
        DATA_FILE_NAME   = self.data_var.get().strip()
        SCHEDULE_FILE_NAME = self.sched_var.get().strip()
        RSM_PATH         = self.rsm_var.get().strip()

        # path validation
        if not os.path.isfile(ECLRUN_EXE):
            messagebox.showerror("Error", f"ECLRUN_EXE not found:\n{ECLRUN_EXE}"); return
        if not os.path.isdir(BASE_DIR):
            messagebox.showerror("Error", f"BASE_DIR not found:\n{BASE_DIR}"); return
        if not os.path.isfile(os.path.join(BASE_DIR, DATA_FILE_NAME)):
            messagebox.showerror("Error", f"DATA file not found:\n{os.path.join(BASE_DIR, DATA_FILE_NAME)}"); return
        if not os.path.isfile(os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)):
            messagebox.showerror("Error", f"SCHEDULE file not found:\n{os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)}"); return

        # build full search-space dict — 4 BHP + 8 injector bounds
        bounds = {}

        for i in range(N_PROD):
            lo, hi = self.bhp_min[i].get(), self.bhp_max[i].get()
            if lo >= hi:
                messagebox.showerror("Error", f"BHP{i+1}: min must be < max"); return
            bounds[f"BHP{i+1}_min"] = lo
            bounds[f"BHP{i+1}_max"] = hi

        for i in range(N_INJ):
            lo, hi = self.inj_min[i].get(), self.inj_max[i].get()
            if lo >= hi:
                messagebox.showerror("Error", f"INJ_RATE{i+1}: min must be < max"); return
            bounds[f"inj{i+1}_min"] = lo
            bounds[f"inj{i+1}_max"] = hi

        n_trials = self.trials_var.get()
        seed     = self.seed_var.get()
        algo     = self.algo_var.get()

        if n_trials <= 0:
            messagebox.showerror("Error", "n_trials must be > 0"); return

        # run in background thread so UI stays responsive
        self.stop_flag["stop"] = False
        self.run_btn.config(state="disabled")
        self.stop_btn.config(state="normal")

        def worker():
            try:
                run_optuna(bounds, algo, n_trials, seed, self.log_write, self.stop_flag)
            except Exception as e:
                self.log_write(f"\nRun failed: {e}\n")
            finally:
                self.run_btn.config(state="normal")
                self.stop_btn.config(state="disabled")

        self.worker = threading.Thread(target=worker, daemon=True)
        self.worker.start()


if __name__ == "__main__":
    app = App()
    app.mainloop()

In [3]:
"""
TKFRONT GUI for Eclipse + Optuna optimization

What it does
- Lets you set search-space bounds for: BHP1-BHP4 (4 producers), inj_rate1-inj_rate8 (8 injectors)
- Lets you choose optimization algorithm:
    * TPE (single-objective: maximize FOPT)
    * NSGA-II (multi-objective: maximize FOPT, minimize FWPT)
- Lets you set number of trials + seed
- Runs optimization and shows best results / Pareto front summary

Requirements
- pip install optuna
- Eclipse runnable via ECLRUN_EXE and files in BASE_DIR
"""

import os
import threading
import subprocess
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import optuna


# ----------------------------
# Defaults (your paths)
# ----------------------------
ECLRUN_EXE = r"C:\ecl\macros\eclrun.exe"
BASE_DIR = r"C:\Users\hp\Desktop\Eclipse"
DATA_FILE_NAME = "Egg_Model_ECL.DATA"
SCHEDULE_FILE_NAME = "SCHEDULE_NEW.INC"
RSM_PATH = os.path.join(BASE_DIR, "EGG_MODEL_ECL_OVERALL.RSM")

N_INJ = 8
N_PROD = 4


# ----------------------------
# Core simulation functions
# ----------------------------
def modify_files(params):
    # --------------------------------------------------
    # Modify SCHEDULE file (Injector RATES) — 8 injectors
    # --------------------------------------------------
    sched_path = os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)

    inj_map = {
        "INJECT1": params["inj_rate1"],
        "INJECT2": params["inj_rate2"],
        "INJECT3": params["inj_rate3"],
        "INJECT4": params["inj_rate4"],
        "INJECT5": params["inj_rate5"],
        "INJECT6": params["inj_rate6"],
        "INJECT7": params["inj_rate7"],
        "INJECT8": params["inj_rate8"],
    }

    with open(sched_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()
        for well, rate in inj_map.items():
            if stripped.startswith(f"'{well}'") and "'RATE'" in stripped:
                parts = stripped.split()
                if "'RATE'" in parts:
                    rate_idx = parts.index("'RATE'") + 1
                    if rate_idx < len(parts):
                        parts[rate_idx] = str(int(rate))
                        line = " ".join(parts) + "\n"
        new_lines.append(line)

    with open(sched_path, "w") as f:
        f.writelines(new_lines)

    # --------------------------------------------------
    # Modify DATA file (Producer BHP) — 4 producers
    # --------------------------------------------------
    data_path = os.path.join(BASE_DIR, DATA_FILE_NAME)

    prod_map = {
        "PROD1": params["BHP1"],
        "PROD2": params["BHP2"],
        "PROD3": params["BHP3"],
        "PROD4": params["BHP4"],
    }

    with open(data_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()
        for well, bhp in prod_map.items():
            if stripped.startswith(f"'{well}'") and "'BHP'" in stripped:
                parts = stripped.split()
                # last token ends with "/" in your file
                parts[-1] = f"{int(bhp)}/"
                line = " ".join(parts) + "\n"
        new_lines.append(line)

    with open(data_path, "w") as f:
        f.writelines(new_lines)


def run_eclipse():
    cmd = [ECLRUN_EXE, "eclipse", DATA_FILE_NAME]
    subprocess.run(cmd, cwd=BASE_DIR, check=True)


def extract_FOPT_FWPT_from_RSM(rsm_path):
    with open(rsm_path, "r") as f:
        lines = f.readlines()

    inside_summary = False
    fopt_idx = fwpt_idx = None

    for line in lines:
        if "SUMMARY OF RUN" in line:
            inside_summary = True
            fopt_idx = fwpt_idx = None
            continue

        if not inside_summary:
            continue

        tokens = line.split()
        if not tokens:
            continue

        # header detection
        if fopt_idx is None and "FOPT" in tokens and "FWPT" in tokens:
            fopt_idx = tokens.index("FOPT")
            fwpt_idx = tokens.index("FWPT")
            continue

        if fopt_idx is None:
            continue

        # numeric rows begin with time
        if not tokens[0][0].isdigit():
            continue

        if float(tokens[0]) == 3600.0:
            return float(tokens[fopt_idx]), float(tokens[fwpt_idx])

    return None, None


# ----------------------------
# Optuna objective factory
# ----------------------------
def make_objective(search_space, mode):
    """
    mode:
      - "tpe_single"  : return fopt (maximize)
      - "nsga2_multi" : return (fopt, fwpt) (maximize, minimize)
    """
    def objective(trial):
        # 4 producer BHP controls
        params = {
            "BHP1": trial.suggest_int("BHP1", search_space["BHP1_min"], search_space["BHP1_max"]),
            "BHP2": trial.suggest_int("BHP2", search_space["BHP2_min"], search_space["BHP2_max"]),
            "BHP3": trial.suggest_int("BHP3", search_space["BHP3_min"], search_space["BHP3_max"]),
            "BHP4": trial.suggest_int("BHP4", search_space["BHP4_min"], search_space["BHP4_max"]),
            # 8 injector rate controls
            "inj_rate1": trial.suggest_int("inj_rate1", search_space["inj1_min"], search_space["inj1_max"]),
            "inj_rate2": trial.suggest_int("inj_rate2", search_space["inj2_min"], search_space["inj2_max"]),
            "inj_rate3": trial.suggest_int("inj_rate3", search_space["inj3_min"], search_space["inj3_max"]),
            "inj_rate4": trial.suggest_int("inj_rate4", search_space["inj4_min"], search_space["inj4_max"]),
            "inj_rate5": trial.suggest_int("inj_rate5", search_space["inj5_min"], search_space["inj5_max"]),
            "inj_rate6": trial.suggest_int("inj_rate6", search_space["inj6_min"], search_space["inj6_max"]),
            "inj_rate7": trial.suggest_int("inj_rate7", search_space["inj7_min"], search_space["inj7_max"]),
            "inj_rate8": trial.suggest_int("inj_rate8", search_space["inj8_min"], search_space["inj8_max"]),
        }

        modify_files(params)

        try:
            run_eclipse()
        except subprocess.CalledProcessError:
            if mode == "nsga2_multi":
                return -1e12, 1e12
            return -1e12

        fopt, fwpt = extract_FOPT_FWPT_from_RSM(RSM_PATH)
        if fopt is None or fwpt is None:
            if mode == "nsga2_multi":
                return -1e12, 1e12
            return -1e12

        if mode == "nsga2_multi":
            return fopt, fwpt
        return fopt

    return objective


# ----------------------------
# Optuna runner
# ----------------------------
def run_optuna(search_space, algo, n_trials, seed, log_fn, stop_flag):
    """
    algo:
      - "TPE (single: max FOPT)"
      - "NSGA-II (multi: max FOPT, min FWPT)"
    """
    if algo.startswith("TPE"):
        sampler = optuna.samplers.TPESampler(
            n_startup_trials=10,
            multivariate=True,
            seed=seed,
        )
        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            study_name="ECLIPSE_TPE_FOPT",
        )
        objective = make_objective(search_space, mode="tpe_single")

    else:
        sampler = optuna.samplers.NSGAIISampler(
            population_size=20,
            mutation_prob=0.2,
            crossover_prob=0.9,
            seed=seed,
        )
        study = optuna.create_study(
            directions=["maximize", "minimize"],
            sampler=sampler,
            study_name="ECLIPSE_NSGA2_FOPT_FWPT",
        )
        objective = make_objective(search_space, mode="nsga2_multi")

    def callback(study, trial):
        if stop_flag["stop"]:
            raise optuna.exceptions.StudyStop()
        if trial.values is None:
            return
        if len(trial.values) == 1:
            log_fn(f"Trial {trial.number:4d}: FOPT={trial.values[0]:.6g}  params={trial.params}\n")
        else:
            log_fn(
                f"Trial {trial.number:4d}: FOPT={trial.values[0]:.6g}  "
                f"FWPT={trial.values[1]:.6g}  params={trial.params}\n"
            )

    log_fn(f"Starting optimization: algo={algo}, n_trials={n_trials}, seed={seed}\n")
    try:
        study.optimize(objective, n_trials=n_trials, callbacks=[callback], show_progress_bar=False)
    except optuna.exceptions.StudyStop:
        log_fn("Stopped by user.\n")
    except Exception as e:
        log_fn(f"ERROR: {e}\n")
        raise

    # summarize
    if study.best_trials:
        log_fn("\n=== Results ===\n")
        if algo.startswith("TPE"):
            bt = study.best_trial
            log_fn(f"Best FOPT : {bt.value:.6g}\n")
            log_fn(f"Best params: {bt.params}\n")
        else:
            pareto = study.best_trials
            log_fn(f"Pareto front size: {len(pareto)}\n")
            for i, t in enumerate(pareto[:10], start=1):
                log_fn(
                    f"Pareto {i:02d}: FOPT={t.values[0]:.6g}  "
                    f"FWPT={t.values[1]:.6g}  params={t.params}\n"
                )

    return study


# ----------------------------
# TKFRONT GUI
# ----------------------------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Eclipse Optimization - TKFRONT")
        self.geometry("820x950")
        self.minsize(820, 700)

        self.stop_flag = {"stop": False}
        self.worker = None

        # ── Paths ──────────────────────────────────────────────────
        paths = ttk.LabelFrame(self, text="Paths")
        paths.pack(fill="x", padx=10, pady=6)

        self.eclrun_var  = tk.StringVar(value=ECLRUN_EXE)
        self.base_var    = tk.StringVar(value=BASE_DIR)
        self.data_var    = tk.StringVar(value=DATA_FILE_NAME)
        self.sched_var   = tk.StringVar(value=SCHEDULE_FILE_NAME)
        self.rsm_var     = tk.StringVar(value=RSM_PATH)

        self._row_entry(paths, 0, "ECLRUN_EXE:",        self.eclrun_var, browse=True, browse_type="file")
        self._row_entry(paths, 1, "BASE_DIR:",          self.base_var,  browse=True, browse_type="dir")
        self._row_entry(paths, 2, "DATA_FILE_NAME:",    self.data_var)
        self._row_entry(paths, 3, "SCHEDULE_FILE_NAME:",self.sched_var)
        self._row_entry(paths, 4, "RSM_PATH:",          self.rsm_var,   browse=True, browse_type="file")

        # ── Search Space ───────────────────────────────────────────
        space = ttk.LabelFrame(self, text="Search Space (integer bounds)")
        space.pack(fill="x", padx=10, pady=6)

        # 4 producer BHP bounds
        self.bhp_min = [tk.IntVar(value=290) for _ in range(N_PROD)]
        self.bhp_max = [tk.IntVar(value=370) for _ in range(N_PROD)]

        # 8 injector rate bounds
        self.inj_min = [tk.IntVar(value=70)  for _ in range(N_INJ)]
        self.inj_max = [tk.IntVar(value=290) for _ in range(N_INJ)]

        # Producer BHP rows
        prod_frame = ttk.LabelFrame(space, text="Producers — BHP (bar)")
        prod_frame.pack(fill="x", padx=6, pady=4)
        for i in range(N_PROD):
            self._bounds_row(prod_frame, i, f"BHP{i+1}", self.bhp_min[i], self.bhp_max[i])

        # Injector rate rows
        inj_frame = ttk.LabelFrame(space, text="Injectors — Rate (sm³/day)")
        inj_frame.pack(fill="x", padx=6, pady=4)
        for i in range(N_INJ):
            self._bounds_row(inj_frame, i, f"INJ_RATE{i+1}", self.inj_min[i], self.inj_max[i])

        # ── Optimization Settings ──────────────────────────────────
        opt = ttk.LabelFrame(self, text="Optimization Settings")
        opt.pack(fill="x", padx=10, pady=6)

        ttk.Label(opt, text="Algorithm:").grid(row=0, column=0, sticky="e", padx=6, pady=6)
        self.algo_var = tk.StringVar(value="NSGA-II (multi: max FOPT, min FWPT)")
        ttk.Combobox(
            opt, textvariable=self.algo_var, state="readonly", width=38,
            values=[
                "TPE (single: max FOPT)",
                "NSGA-II (multi: max FOPT, min FWPT)",
            ],
        ).grid(row=0, column=1, sticky="w", padx=6, pady=6)

        self.trials_var = tk.IntVar(value=110)
        self.seed_var   = tk.IntVar(value=42)

        ttk.Label(opt, text="n_trials:").grid(row=1, column=0, sticky="e", padx=6, pady=6)
        ttk.Entry(opt, textvariable=self.trials_var, width=12).grid(row=1, column=1, sticky="w", padx=6, pady=6)
        ttk.Label(opt, text="seed:").grid(row=1, column=2, sticky="e", padx=6, pady=6)
        ttk.Entry(opt, textvariable=self.seed_var,   width=12).grid(row=1, column=3, sticky="w", padx=6, pady=6)
        opt.grid_columnconfigure(4, weight=1)

        # ── Buttons ────────────────────────────────────────────────
        btns = ttk.Frame(self)
        btns.pack(fill="x", padx=10, pady=6)

        self.run_btn  = ttk.Button(btns, text="Run Optimization", command=self.on_run)
        self.run_btn.pack(side="left")
        self.stop_btn = ttk.Button(btns, text="Stop", command=self.on_stop, state="disabled")
        self.stop_btn.pack(side="left", padx=10)
        self.clear_btn = ttk.Button(btns, text="Clear Log", command=self.clear_log)
        self.clear_btn.pack(side="left")

        # ── Log ────────────────────────────────────────────────────
        logf = ttk.LabelFrame(self, text="Log")
        logf.pack(fill="both", expand=True, padx=10, pady=6)

        sb = ttk.Scrollbar(logf)
        sb.pack(side="right", fill="y")
        self.log = tk.Text(
            logf,
            wrap="word",
            yscrollcommand=sb.set,
            font=("Consolas", 11),   # larger monospace font
            height=20,               # minimum visible rows
            padx=8,
            pady=6,
        )
        self.log.pack(fill="both", expand=True)
        sb.config(command=self.log.yview)

        self.log_write("Ready.\n")

    # ── Helpers ────────────────────────────────────────────────────
    def _row_entry(self, parent, row, label, var, browse=False, browse_type="file"):
        ttk.Label(parent, text=label).grid(row=row, column=0, sticky="e", padx=6, pady=3)
        ttk.Entry(parent, textvariable=var, width=80).grid(row=row, column=1, sticky="we", padx=6, pady=3)
        parent.grid_columnconfigure(1, weight=1)
        if browse:
            def do_browse(v=var, bt=browse_type):
                path = filedialog.askopenfilename() if bt == "file" else filedialog.askdirectory()
                if path:
                    v.set(path)
            ttk.Button(parent, text="Browse", command=do_browse).grid(row=row, column=2, padx=6, pady=3)

    def _bounds_row(self, parent, row, name, vmin, vmax):
        ttk.Label(parent, text=f"{name} min:").grid(row=row, column=0, sticky="e", padx=6, pady=3)
        ttk.Entry(parent, textvariable=vmin, width=10).grid(row=row, column=1, sticky="w", padx=6, pady=3)
        ttk.Label(parent, text=f"{name} max:").grid(row=row, column=2, sticky="e", padx=6, pady=3)
        ttk.Entry(parent, textvariable=vmax, width=10).grid(row=row, column=3, sticky="w", padx=6, pady=3)
        parent.grid_columnconfigure(4, weight=1)

    def log_write(self, msg: str):
        self.log.insert("end", msg)
        self.log.see("end")
        self.update_idletasks()

    def clear_log(self):
        self.log.delete("1.0", "end")

    def on_stop(self):
        self.stop_flag["stop"] = True
        self.log_write("\nStop requested...\n")

    # ── Run ────────────────────────────────────────────────────────
    def on_run(self):
        # apply globals from UI
        global ECLRUN_EXE, BASE_DIR, DATA_FILE_NAME, SCHEDULE_FILE_NAME, RSM_PATH
        ECLRUN_EXE       = self.eclrun_var.get().strip()
        BASE_DIR         = self.base_var.get().strip()
        DATA_FILE_NAME   = self.data_var.get().strip()
        SCHEDULE_FILE_NAME = self.sched_var.get().strip()
        RSM_PATH         = self.rsm_var.get().strip()

        # path validation
        if not os.path.isfile(ECLRUN_EXE):
            messagebox.showerror("Error", f"ECLRUN_EXE not found:\n{ECLRUN_EXE}"); return
        if not os.path.isdir(BASE_DIR):
            messagebox.showerror("Error", f"BASE_DIR not found:\n{BASE_DIR}"); return
        if not os.path.isfile(os.path.join(BASE_DIR, DATA_FILE_NAME)):
            messagebox.showerror("Error", f"DATA file not found:\n{os.path.join(BASE_DIR, DATA_FILE_NAME)}"); return
        if not os.path.isfile(os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)):
            messagebox.showerror("Error", f"SCHEDULE file not found:\n{os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)}"); return

        # build full search-space dict — 4 BHP + 8 injector bounds
        bounds = {}

        for i in range(N_PROD):
            lo, hi = self.bhp_min[i].get(), self.bhp_max[i].get()
            if lo >= hi:
                messagebox.showerror("Error", f"BHP{i+1}: min must be < max"); return
            bounds[f"BHP{i+1}_min"] = lo
            bounds[f"BHP{i+1}_max"] = hi

        for i in range(N_INJ):
            lo, hi = self.inj_min[i].get(), self.inj_max[i].get()
            if lo >= hi:
                messagebox.showerror("Error", f"INJ_RATE{i+1}: min must be < max"); return
            bounds[f"inj{i+1}_min"] = lo
            bounds[f"inj{i+1}_max"] = hi

        n_trials = self.trials_var.get()
        seed     = self.seed_var.get()
        algo     = self.algo_var.get()

        if n_trials <= 0:
            messagebox.showerror("Error", "n_trials must be > 0"); return

        # run in background thread so UI stays responsive
        self.stop_flag["stop"] = False
        self.run_btn.config(state="disabled")
        self.stop_btn.config(state="normal")

        def worker():
            try:
                run_optuna(bounds, algo, n_trials, seed, self.log_write, self.stop_flag)
            except Exception as e:
                self.log_write(f"\nRun failed: {e}\n")
            finally:
                self.run_btn.config(state="normal")
                self.stop_btn.config(state="disabled")

        self.worker = threading.Thread(target=worker, daemon=True)
        self.worker.start()


if __name__ == "__main__":
    app = App()
    app.mainloop()